In [3]:
from warnings import filterwarnings
filterwarnings('ignore')
import pandas as pd
dividend = pd.read_parquet('data/dividend.parquet')
dividend = dividend.sort_values(by=['stock_code', 'ex_date'], ascending=[True, True])
dividend.head(10)

,stock_code,announce_date,ex_date,div_pre_tax,div_after_tax,progress,base_shares,div_year,currency,is_no_div,total_cash_div
8165,000001.SZ,20130614,20130620,0.170001,0.131472,3,5.123809e+05,20121231,CNY,0,8.709306e+08
6411,000001.SZ,20140606,20140612,0.159974,0.151981,3,9.521493e+05,20131231,CNY,0,1.523397e+09
20151,000001.SZ,20150407,20150413,0.173981,0.165298,3,1.142486e+06,20141231,CNY,0,1.988210e+09
21055,000001.SZ,20160608,20160616,0.152994,0.152997,3,1.430813e+06,20151231,CNY,0,2.189083e+09
3884,000001.SZ,20170717,20170721,0.157993,0.158005,3,1.717085e+06,20161231,CNY,0,2.712675e+09
6258,000001.SZ,20180706,20180712,0.135999,0.136014,3,1.717202e+06,20171231,CNY,0,2.335101e+09
16687,000001.SZ,20190620,20190626,0.144977,0.145004,3,1.717122e+06,20181231,CNY,0,2.489544e+09
13074,000001.SZ,20200522,20200528,0.218038,0.218027,3,1.940372e+06,20191231,CNY,0,4.230812e+09
26102,000001.SZ,20210507,20210514,0.179976,0.179985,3,1.940534e+06,20201231,CNY,0,3.493105e+09
21192,000001.SZ,20220715,20220722,0.227990,0.227989,3,1.940405e+06,20211231,CNY,0,4.425158e+09


预期股息率（TTM）：最近365个自然日div_pre_tax和 / **不复权**市场价格，计入日期区间所有announce_date在这之间的总分红；
静态股息率：上一财年总div_pre_tax和 / **不复权**市场价格，计入ex_date在当前日期之前且div_year为当前日期上一年的总分红；
动态股息率（TTM）：最近365个自然日div_pre_tax和 / **不复权**市场价格，计入日期区间所有ex_date在这之间的总分红；
~~div_after_tax~~ div_pre_tax ：税前股息反映的是公司真实分红能力，税后股息只反映单个投资者的到手收益，前者更适合做因子/估值。

In [4]:
price_dt = pd.read_parquet('data/eod_prices.parquet')
price_dt=price_dt.sort_values(by=['stock_code', 'trade_date'], ascending=[True, True])
price_dt.head()


,stock_code,trade_date,close_price,adjusted_close,adj_factor,volume,turnover
2983,000001.SZ,20130104,15.99,355.0546,22.204789,443851.37,717567.5466
8347,000001.SZ,20130107,16.30,361.9381,22.204789,357169.25,578450.4876
10530,000001.SZ,20130108,16.00,355.2766,22.204789,312479.12,501360.0937
13553,000001.SZ,20130109,15.86,352.1680,22.204789,251329.15,399696.1831
10100,000001.SZ,20130110,15.87,352.3900,22.204789,240030.27,383347.7326


In [7]:
import pandas as pd
from util import calc_factors
from tqdm import tqdm

dividend['announce_date'] = pd.to_datetime(dividend['announce_date'])
dividend['ex_date'] = pd.to_datetime(dividend['ex_date'])
price_dt['trade_date'] = pd.to_datetime(price_dt['trade_date'])

price_dt = price_dt.sort_values(['stock_code', 'trade_date'])
dividend = dividend.sort_values(['stock_code', 'announce_date'])

df_factors_list = []

# 获取总股票数
total_stocks = price_dt['stock_code'].nunique()

# 使用 tqdm 包装 groupby 迭代
for stock, price_sub in tqdm(price_dt.groupby('stock_code'), total=total_stocks, desc="Processing stocks"):
    if stock in dividend['stock_code'].values:
        div_sub = dividend[dividend['stock_code'] == stock]
        factor_sub = calc_factors(price_sub, div_sub)
        df_factors_list.append(factor_sub)
    else:
        price_sub = price_sub.copy()
        price_sub['expected_div_yield'] = 0.0
        price_sub['static_div_yield'] = 0.0
        price_sub['dynamic_div_yield'] = 0.0
        df_factors_list.append(price_sub[['stock_code', 'trade_date', 
                                          'expected_div_yield', 'static_div_yield', 'dynamic_div_yield']])

df_factors = pd.concat(df_factors_list, ignore_index=True)

Processing stocks: 100%|██████████| 5693/5693 [3:58:00<00:00,  2.51s/it]  


In [ ]:
df_factors.tail(20)

,stock_code,trade_date,expected_div_yield,static_div_yield,dynamic_div_yield
12065627,920992.BJ,2025-11-03,0.003737,0.0,0.003737
12065628,920992.BJ,2025-11-04,0.003795,0.0,0.003795
12065629,920992.BJ,2025-11-05,0.003756,0.0,0.003756
12065630,920992.BJ,2025-11-06,0.003830,0.0,0.003830
12065631,920992.BJ,2025-11-07,0.003901,0.0,0.003901
12065632,920992.BJ,2025-11-10,0.003885,0.0,0.003885
12065633,920992.BJ,2025-11-11,0.003937,0.0,0.003937
12065634,920992.BJ,2025-11-12,0.003920,0.0,0.003920
12065635,920992.BJ,2025-11-13,0.003918,0.0,0.003918
12065636,920992.BJ,2025-11-14,0.003947,0.0,0.003947


In [15]:
import pandas as pd
from scipy.stats.mstats import winsorize
import os

# ===================== 1. 数据预处理 =====================
# 假设 df_factors 已存在，包含列：stock_code, trade_date, dynamic_div_yield
# 确保 trade_date 为 datetime 类型（如果不是则转换）
df_factors['trade_date'] = pd.to_datetime(df_factors['trade_date'])

# 可选：去除股息率为 NaN 或 0 的股票（根据业务逻辑决定）
# df_factors = df_factors[df_factors['dynamic_div_yield'] > 0]

# ===================== 2. 核心选股逻辑 =====================
def select_stocks_by_date(date_group):
    """
    对单个日期的样本执行选股：
    1. （可选）对 dynamic_div_yield 做 1%/99% 缩尾
    2. 按缩尾后股息率降序排序，取前 50 只
    """
    trade_date = date_group.name
    daily_data = date_group.copy()
    
    # 剔除股息率为空或无效的股票
    daily_data = daily_data.dropna(subset=['dynamic_div_yield'])
    
    if len(daily_data) == 0:
        return pd.DataFrame()
    
    # 可选：缩尾处理（1%和99%分位数）
    # 如果不需要缩尾，可注释下面两行，直接用原始 dynamic_div_yield
    daily_data['div_yield_winsorized'] = winsorize(
        daily_data['dynamic_div_yield'].values,
        limits=(0.01, 0.01),
        inclusive=(True, True)
    )
    
    # 按缩尾后股息率降序排序，取前50只
    selected = daily_data.sort_values('div_yield_winsorized', ascending=False).head(50)
    
    # 添加排名（整体股息率从高到低）
    selected = selected.reset_index(drop=True)
    selected['selection_rank'] = range(1, len(selected) + 1)
    
    return selected

# 按交易日分组执行选股
selected_stocks = df_factors.groupby('trade_date', group_keys=False).apply(select_stocks_by_date)

# 重置索引（清理groupby后的索引）
selected_stocks = selected_stocks.reset_index(drop=True)

# 保留核心列（可自行调整）
final_selection_df = selected_stocks[['trade_date', 'stock_code', 'dynamic_div_yield', 'div_yield_winsorized', 'selection_rank']]

# ===================== 3. 保存结果 =====================
# 确保保存目录存在
os.makedirs('records', exist_ok=True)

final_selection_df.to_excel('records/div_selection.xlsx', index=False)

print(f"选股完成，共 {len(final_selection_df)} 条记录，保存至 records/div_selection.xlsx")

选股完成，共 156700 条记录，保存至 records/div_selection.xlsx


In [17]:
final_selection_df.tail()

,trade_date,stock_code,dynamic_div_yield,div_yield_winsorized,selection_rank
156695,2025-11-28,601886.SH,0.068927,0.063729,46
156696,2025-11-28,002327.SZ,0.085155,0.063729,47
156697,2025-11-28,605377.SH,0.079991,0.063729,48
156698,2025-11-28,605368.SH,0.088999,0.063729,49
156699,2025-11-28,600566.SH,0.079589,0.063729,50


回测时使用复权价格，检查：因子计算使用原始价格

In [22]:
import pandas as pd
import numpy as np
from util import *

# ----------------------------
# User‑configurable parameters
# ----------------------------
INITIAL_CASH = 1_000_000        # initial capital
START_DATE = '2017-01-01'       # backtest start date (inclusive)
END_DATE   = '2025-06-30'       # backtest end date (inclusive)
REBALANCE_FREQ = 20             # rebalance every N calendar days (using available selection dates)
TOP_N = 20                       # number of top‑ranked stocks to hold each rebalance

PRICE_FILE = 'data/eod_prices.parquet'                 # path to price file
SELECTION_FILE = 'records/div_selection.xlsx'          # path to dividend factor selection file

# ----------------------------
# Load and prepare data
# ----------------------------
print("Loading data...")
price_df = load_and_preprocess_price(PRICE_FILE)
selection_df = load_selection(SELECTION_FILE)

# Create pivot table: rows = dates, columns = stocks, values = adjusted_close
price_pivot = price_df.pivot(index='trade_date', columns='stock_code', values='adjusted_close')
price_pivot = price_pivot.sort_index()
# Ensure index is datetime (fix for categorical index)
price_pivot.index = pd.to_datetime(price_pivot.index)

# Filter by date range
price_pivot = price_pivot.loc[START_DATE:END_DATE]
selection_df = selection_df[(selection_df['trade_date'] >= START_DATE) &
                            (selection_df['trade_date'] <= END_DATE)]

# Get sorted unique selection dates (these are the only dates we consider for rebalancing)
selection_dates = sorted(selection_df['trade_date'].unique())
if not selection_dates:
    raise ValueError("No selection data available in the given date range.")

# Determine rebalance dates (every REBALANCE_FREQ-th date in the selection_dates list)
rebalance_dates = selection_dates[::REBALANCE_FREQ]
print(f"Number of rebalance dates: {len(rebalance_dates)}")

# Generate the full timeline (all trading days in the pivot index)
all_dates = price_pivot.index.tolist()
# Ensure we start from the first rebalance date (before that we have no positions)
first_rebalance = rebalance_dates[0]
timeline = [d for d in all_dates if d >= first_rebalance]

# ----------------------------
# Backtest loop
# ----------------------------
cash = INITIAL_CASH
holdings = {}  # stock_code -> shares
portfolio_values = []
benchmark_cumulative = None

# Pre‑compute benchmark returns (daily equal‑weighted average)
benchmark_daily, benchmark_cum = compute_benchmark_returns(price_pivot, START_DATE, END_DATE)

# For each day in timeline
for date in timeline:
    # Get today's prices for all stocks (if missing, price will be NaN)
    today_prices = price_pivot.loc[date]

    # --- Rebalance if today is a rebalance date ---
    if date in rebalance_dates:
        # 1. Sell all current holdings
        for stock, shares in holdings.items():
            price = today_prices.get(stock, np.nan)
            if pd.notna(price):
                cash += shares * price
            # If price is missing, we assume the stock is worthless (sell for 0)
        holdings = {}

        # 2. Get selected stocks for today (dividend factor top N)
        selected = get_top_n_selection(selection_df, date, TOP_N)

        # 3. Buy new holdings equally
        if selected:
            # Filter stocks that have a valid price today
            valid_stocks = [s for s in selected if pd.notna(today_prices.get(s, np.nan))]
            if valid_stocks:
                amount_per_stock = cash / len(valid_stocks)
                for stock in valid_stocks:
                    price = today_prices[stock]
                    shares = amount_per_stock / price
                    holdings[stock] = shares
                cash = 0.0  # all cash invested
            else:
                # No valid stocks – keep cash
                pass

    # --- Calculate total portfolio value today ---
    portfolio_value = cash
    for stock, shares in holdings.items():
        price = today_prices.get(stock, np.nan)
        if pd.notna(price):
            portfolio_value += shares * price
        # If price is missing, the position contributes 0
    portfolio_values.append(portfolio_value)

# ----------------------------
# Prepare results
# ----------------------------
portfolio_series = pd.Series(portfolio_values, index=timeline, name='Portfolio')

benchmark_series = benchmark_cum.reindex(timeline, method='ffill').fillna(1.0)
benchmark_funds = INITIAL_CASH * benchmark_series

portfolio_returns = portfolio_series.pct_change().dropna()

metrics = compute_metrics(portfolio_returns, rf=0, periods_per_year=252)
print("\n--- Performance Metrics (Dividend Factor) ---")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

plot_results(portfolio_series, benchmark_funds, timeline,
             title='Backtest: Dividend Factor – Top {} Stocks Rebalanced Every {} Days'.format(TOP_N, REBALANCE_FREQ))

Loading data...
Number of rebalance dates: 103

--- Performance Metrics (Dividend Factor) ---
Annualized Return: 0.0754
Volatility: 0.2031
Sharpe Ratio: 0.3711
Max Drawdown: -0.3182


股息因子长期较为有效。股息因子在2020年1月1日之后的收益比较夸张。可以尝试预期股息因子的回测。